In [ ]:
from dotenv import load_dotenv
from pathlib import Path, PurePath, __file__
import os
import logging
from joblib import dump, load
from typing import Tuple, List, Dict, Any
import pandas as pd
import nflreadpy as nfl
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, KFold, StratifiedKFold
from sklearn.ensemble import (RandomForestRegressor,HistGradientBoostingRegressor)
from sklearn.metrics import mean_squared_error, r2_score


# Locations
BACKEND_DIR = Path(__file__).parent
BASE_DIR = BACKEND_DIR.parent
DATA_DIR = BACKEND_DIR / "data"
MODELS_DIR = BACKEND_DIR / "models"
LOG_DIR = BACKEND_DIR / "logs"
FRONTEND_DIR = BASE_DIR / "frontend"
FRONTEND_DIST = FRONTEND_DIR / "dist"
FRONTEND_BUILD = FRONTEND_DIST  # Alias for compatibility

# Truthy parsing helper
TRUTHY = {"true", "t", "1", "yes", "y"}

def _load_env() -> None:
    """
    Load .env from backend or repo root.
    """
    dotenv_loaded = load_dotenv(BACKEND_DIR / ".env")

    if not dotenv_loaded:
        load_dotenv(BASE_DIR / ".env")
        print("Loaded .env from repo root")
        return load_dotenv(BASE_DIR / ".env")
    print(f"Loaded .env from backend: {dotenv_loaded}")
    return dotenv_loaded

print(f"Loading .env...")
_load_env()
print(f".env loaded.{os.listdir(BACKEND_DIR)}")

In [ ]:
import math
from build_csv_datasetsv3 import build_dataset
from backend.utils import   

# ---------------------------------------------------------------------
# Load Models from disk
# ---------------------------------------------------------------------

clf_path = MODELS_DIR / "histgradient_home_win_clf.joblib"
log = logging.getLogger("backend.main")

def _predict_home_win_prob():
    """Predict home win probability using histgradient classifier with fallback to logistic function."""
    clf = load(clf_path)
    df = build_dataset(2023, 2023, DATA_DIR / "predictions_temp")


    # Try direct prediction
    if hasattr(clf, "predict_proba"):
        try:
            proba = clf.predict_proba(X_raw)
            idx = _pick_positive_class_index(clf)
            p = float(proba[0][idx])
            return float(np.clip(p, 0.0, 1.0)), False
        except Exception as e:
            log.warning("[Predict] hist_win_clf predict_proba(raw) failed: %s", e)

        # Try transformed prediction
        try:
            X_tx = bundle.preprocessor.transform(_safe_fill(X_raw))
            proba = clf.predict_proba(X_tx)
            idx = _pick_positive_class_index(clf)
            p = float(proba[0][idx])
            return float(np.clip(p, 0.0, 1.0)), False
        except Exception as e:
            log.warning("[Predict] hist_win_clf predict_proba(preprocessed) failed: %s", e)

    # Fallback to logistic function
    p = 1.0 / (1.0 + math.exp(-0.25 * float(point_diff)))
    return float(np.clip(p, 0.0, 1.0)), True


In [26]:
import pandas as pd
import os
import requests
from pydantic import BaseModel
from typing import Dict, Any

class TeamCard(BaseModel):
    key: str
    value: Any
   


logo_df = pd.read_csv("../backend/data/team_logos.csv")
print(logo_df.head())

for key, value in logo_df.iterrows():
  print(key, value)
  team_card = TeamCard(key=str(key), value=value)
  print(team_card)  

  team_abbr          team_name  team_id  team_nick team_conf team_division  \
0       ARI  Arizona Cardinals     3800  Cardinals       NFC      NFC West   
1       ATL    Atlanta Falcons      200    Falcons       NFC     NFC South   
2       BAL   Baltimore Ravens      325     Ravens       AFC     AFC North   
3       BUF      Buffalo Bills      610      Bills       AFC      AFC East   
4       CAR  Carolina Panthers      750   Panthers       NFC     NFC South   

  team_color team_color2 team_color3 team_color4  \
0    #97233F     #000000     #ffb612     #a5acaf   
1    #A71930     #000000     #a5acaf     #a30d2d   
2    #241773     #9E7C0C     #9e7c0c     #c60c30   
3    #00338D     #C60C30     #0c2e82     #d50a0a   
4    #0085CA     #000000     #bfc0bf     #0085ca   

                                 team_logo_wikipedia  \
0  https://upload.wikimedia.org/wikipedia/en/thum...   
1  https://upload.wikimedia.org/wikipedia/en/thum...   
2  https://upload.wikimedia.org/wikipedia/en/thum.